# 2단계 · CosFace 탐침

같은 RUN_ID를 사용합니다. [실행과 결과 안내](HOW_TO_READ.md)

새 Parent 학습 전 TinyFace 대조 탐침은 같은 v3 RUN_ID에서 PARENT_SOURCE="legacy_parent_wd0", PROBE_DATASETS=["tinyface"]로 실행합니다. 40 epoch 후 distractor 제외 Rank-1을 확인합니다.

In [ ]:
# cell 1 : 설정 및 경로 확인
import json
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "fine-tune/modules").is_dir())
sys.path.insert(0, str(repo / "fine-tune"))
from modules.clean_training import final_artifact, recipe, run_directory, run_stage
from modules.clean_campaign import evaluation_dir, model_dir, origin_matrix, parent_validation, read_result, run_full, run_probe, validate_choices

DATA_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets"))
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models"))
RUN_ID = "clean_mfn1024_s8972_v3"
SEED = 8972
EXECUTE = False
RESUME = False
KEEP_RAW_RESULTS = False
PARENT_SOURCE = "clean_parent"
PROBE_DATASETS = ["tinyface", "survface"]
run = run_directory(MODEL_ROOT, RUN_ID)
validate_choices(PROBE_DATASETS, ["cosface"], "paper_table1", PARENT_SOURCE)
paths = [model_dir(run, "paper_table1", PARENT_SOURCE, ds, "cosface") for ds in PROBE_DATASETS]
display(pd.DataFrame({"dataset": PROBE_DATASETS, "path": paths}))

In [ ]:
# cell 2 : 실행
if EXECUTE:
    probe = run_probe(DATA_ROOT, run, PARENT_SOURCE, PROBE_DATASETS,
                      seed=SEED, keep_raw_results=KEEP_RAW_RESULTS, resume=RESUME)

In [ ]:
# cell 3 : 결과 확인
status_rows = []
comparison_rows = []
for ds in PROBE_DATASETS:
    for source in ("clean_parent", "legacy_parent_wd0"):
        folder = model_dir(run, "paper_table1", source, ds, "cosface")
        evaluation = evaluation_dir(folder, ds, probe=True)
        trained = (folder / "COMPLETED").is_file()
        evaluated = (evaluation / "COMPLETED").is_file()
        snapshot = folder / ("summary.json" if trained else "progress.json")
        record = json.loads(snapshot.read_bytes()) if snapshot.is_file() else {}
        epochs = record.get("epochs", [])
        last = epochs[-1] if epochs else {}
        failure = last.get("health_gate_failure")
        if failure:
            status = "health_gate_failed"
        elif trained:
            status = "completed"
        elif record:
            status = "in_progress"
        else:
            status = "pending"
        status_rows.append(dict(
            dataset=ds, parent_source=source, training_status=status, evaluation=evaluated,
            epoch=last.get("epoch"), loss=last.get("loss"), head_direction_norm=last.get("head_direction_norm"),
            output_bn_weight_l2=last.get("output_bn_weight_l2"), health_gate_failure=failure,
        ))
        if evaluated:
            summary = read_result(evaluation)
            comparison_rows.extend(dict(parent_source=source, parent_sha256=summary["parent_sha256"], **row)
                                   for row in summary["comparison"])

display(pd.DataFrame(status_rows).astype({"epoch": "Int64"}))
if comparison_rows:
    display(pd.DataFrame(comparison_rows))